# LS-NEQ 11_transplant — Cross-Modell-Transplantation

**Dossier-Stufe L** (`lsne.experiments.STAGES["L"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** `docs/DESIGN.md` §4.9: modellagnostische Maschinerie mit steckbarem Quelladapter (`lsne.transplant`), validiert mit einem Stand-in-Quellmodell, bevor ein BlockGPT-artiges Spurenmodell eingehängt wird (`docs/BLOCKGPT_FEASIBILITY.md`: es gibt keine BlockGPT-Gewichte; empfohlen ist ein eigenes Modell auf Xatu-2016-Traces, `docs/TRACE_SOURCE_SPEC.md`). Schritte: gepaarte Eingaben (englische/deutsche Einheitssätze der Familien), lineare Abbildung Quelle → Ziel (Procrustes nach PCA) mit Permutations-Null, Layer-Matching, Transfer der im Quellmodell bestimmten Sprachachse und **funktionaler Transfertest** (Kosinus zur nativen Achse, Verschiebung des deutschen Anteils in Fortsetzungen), dann Injektion A (Residual-Substitution, kein Addieren), B (KV-Cache-Pflanzung über ein virtuelles Präfix auf allen Layern) und C (beides) gegen Zufalls-, Identitäts- und Permutationskontrollen.

**Eingaben.** Stimulusbank (Satzpaare, K=0-Prompts), Quellmodell `TRANSPLANT_SOURCE` (`rotated_self`: Standard-Stand-in, `self`, oder ein HF-Modell wie `EleutherAI/pythia-70m@step143000`; ein Spurenmodell über dieselbe `SourceModel`-Schnittstelle), Generierungsbudget `cfg.gen_n_new × cfg.gen_n_samples`.

**Ausgaben.** Tabellen `transfer_test`, `transfer_summary`, `layer_matching`, `alignments`, `transplant`, `transplant_summary`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/L/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/11_transplant/`; am Ende eine kurze `summary_11_transplant_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Stufe-L-Parameter → Modell → Stufe L → Transfertest pro Layer → Layer-Matching → Transplantation A/B/C gegen Kontrollen → Zusammenfassung.

**Präregistrierung.** Kontrast 18 (`transfer_summary`: Alignment-R² Hold-out − permutiert ≥ 0,2 `r2_gap_ok`, Kosinus transferierte vs. native Sprachachse ≥ 0,3 im besten Layer `cos_ok`, Vorzeichengleichheit der f_D-Verschiebung `sign_agreement`, zusammen `validated`; `transplant_summary.sign_agrees_with_native` pro Modus). Erst nach bestandener Stand-in-Validierung wird ein Spurenmodell als Quelle gewertet; dessen „Teilen“-Merkmal wird vorab im Quellmodell definiert und eingefroren.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    # a reused runtime: move the shallow checkout to the branch tip (a fast-forward pull fails on
    # depth-1 history and would silently keep the old code)
    if _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
           f"git -C {REPO_DIR} checkout -q -f -B {GIT_BRANCH} FETCH_HEAD") != 0:
        raise RuntimeError(f"{REPO_DIR} konnte nicht auf den Stand von {GIT_BRANCH} gebracht werden: "
                           "Laufzeit -> Sitzung trennen und löschen, dann erneut ausführen")
elif _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}") != 0:
    raise RuntimeError(f"git clone von {GIT_URL} ({GIT_BRANCH}) fehlgeschlagen")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
# modules imported by an earlier run in this runtime would shadow the updated checkout
for _m in [m for m in sys.modules if m == "lsne" or m.startswith("lsne.")]:
    del sys.modules[_m]
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "11_transplant"
STAGE = "L"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-L-Parameter: Quellmodell ------------------------------------------------------------
TRANSPLANT_SOURCE = None   # "rotated_self" (Standard: Stand-in, das Alignment muss eine Isometrie rückgängig machen) | "self" | "EleutherAI/pythia-70m@step143000"
print("Quellmodell:", TRANSPLANT_SOURCE or limits.l_source, "| Satzpaare je Sprache:", limits.l_n_texts, "| Prompts:", limits.l_n_prompts,
      "| KV-Präfix:", limits.l_n_prefix, "| Dosis:", limits.l_eps, "| Permutations-Null:", limits.l_n_null)
try:
    import lsne.transplant  # noqa: F401
    print("lsne.transplant verfügbar")
except Exception as e:   # the stage then writes a manifest with stub=True and no tables
    print("lsne.transplant NICHT importierbar -> Stufe L schreibt nur ein Manifest (stub):", repr(e))
if TRANSPLANT_SOURCE:
    CLI_COMMAND += f" --transplant-source {TRANSPLANT_SOURCE}"


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def stage_status(stage: str, rd, status: str) -> dict:
    """Report entry of a finished stage: a stale error.txt of an earlier failed run is removed
    (the manifest says ok) and failed substages (G-L, manifest.json -> substages) are listed;
    the status then reads ok_with_substage_errors / reused_with_substage_errors."""
    if C.clear_stale_error(rd.path):
        print(f"Stufe {stage}: veraltete error.txt eines früheren Laufs entfernt ({rd.path})")
    sub = C.substage_status(rd.path, status)
    for name, err in sub["substage_errors"].items():
        print(f"Stufe {stage}: Substufe {name} FEHLER: {err}")
    return sub


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(run_dir=str(rd.path), seconds=0.0, error="", **stage_status(stage, rd, "reused"))
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="",
                             **stage_status(stage, rd, "ok"))
        print(f"Stufe {stage} {report[stage]['status']} in {report[stage]['seconds']:.0f} s: "
              f"{', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe L ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage L --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> \
#       [--transplant-source EleutherAI/pythia-70m@step143000]
res_L = run_stage("L", transplant_source=TRANSPLANT_SOURCE)
print({k: len(v) for k, v in res_L.items()})
manifest_L = substage_report("L")
if manifest_L:
    print("Quelle:", manifest_L.get("source"), "| Satzpaare:", manifest_L.get("n_texts"), "| validiert:", manifest_L.get("validated"))


In [ ]:
# --- Kontrast 18: funktionaler Transfertest pro Layer des Rasters (Stand-in-Validierung) -------
with guarded("Funktionaler Transfertest"):
    tt = table("L", "transfer_test")
    ts = table("L", "transfer_summary")
    if tt.empty:
        print("Keine Transfer-Tabelle (Stub oder Substufe fehlgeschlagen; siehe Substufen-Bericht).")
    else:
        tt = tt.sort_values("layer_tgt")
        fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
        axes[0].plot(tt["layer_tgt"], tt["cos_transferred_vs_native"], "o-", label="transferierte vs. native Sprachachse")
        axes[0].errorbar(tt["layer_tgt"], tt["cos_null_vs_native"], yerr=tt["cos_null_std"].fillna(0), fmt="s--", color="grey",
                         capsize=2, label="permutierte Paare (Null)")
        axes[0].plot(tt["layer_tgt"], tt["cos_null_max"], ":", color="grey", label="Null-Maximum")
        axes[0].axhline(0.3, color="r", lw=0.8, ls="--", label="Mindesteffekt cos ≥ 0,3")
        axes[0].set_title("Kosinus im Zielmodell", fontsize=9)
        axes[1].plot(tt["layer_tgt"], tt["r2_holdout"], "o-", label="R² Hold-out")
        axes[1].plot(tt["layer_tgt"], tt["r2_permuted"], "s--", color="grey", label="R² permutierte Paare")
        axes[1].plot(tt["layer_tgt"], tt["r2_gap"], "^:", color="C2", label="Differenz (Mindestabstand 0,2)")
        axes[1].axhline(0.2, color="r", lw=0.8, ls="--")
        axes[1].set_title("Alignment-Güte (Procrustes nach PCA)", fontsize=9)
        w = 0.4
        axes[2].bar(tt["layer_tgt"] - w / 2, tt["delta_fD_transferred"], w, label="Δf_D transferierte Achse")
        axes[2].bar(tt["layer_tgt"] + w / 2, tt["delta_fD_native"], w, label="Δf_D native Achse")
        axes[2].axhline(0, color="k", lw=0.6)
        axes[2].set_title("Verschiebung des deutschen Anteils in Fortsetzungen", fontsize=9)
        for ax in axes:
            ax.set_xlabel("Ziel-Layer (Block-Eingang)")
            ax.legend(fontsize=7)
        savefig("transfer_test")
        plt.show()
        show(tt[["layer_tgt", "layer_src", "cos_transferred_vs_native", "z_cos_vs_null", "r2_holdout", "r2_permuted", "r2_gap",
                 "sign_agreement", "readout_alive", "r2_gap_ok", "cos_ok", "validated"]])
        if len(ts):
            s = ts.iloc[0]
            print(f"Bester Layer {s['best_layer']}: validiert = {s['validated']} ({s['n_validated_layers']} von {s['n_layers']} Layern). "
                  "Erst nach bestandener Validierung wird ein Spurenmodell als Quelle gewertet.")


In [ ]:
# --- Layer-Matching: R² Hold-out über Quell- × Ziel-Layer; Alignments je Ziel-Layer (KV-Pflanzung) --
with guarded("Layer-Matching"):
    lm = table("L", "layer_matching")
    al = table("L", "alignments")
    piv = lm.pivot_table(index="layer_src", columns="layer_tgt", values="r2_holdout")
    fig, ax = plt.subplots(figsize=(0.6 * piv.shape[1] + 3, 0.5 * piv.shape[0] + 2))
    im = ax.imshow(piv.to_numpy(dtype=float), cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(piv.shape[1]))
    ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels(piv.index)
    ax.set_xlabel("Ziel-Layer (Pythia)")
    ax.set_ylabel("Quell-Layer")
    ax.set_title("R² (Hold-out) der linearen Abbildung Quelle → Ziel", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
    savefig("layer_matching")
    plt.show()
    if len(al):
        print("Alignments je Ziel-Layer (pflanzen das KV-Präfix auf jedem Layer):")
        show(al)


In [ ]:
# --- Transplantation: Residual-Substitution (A), KV-Pflanzung (B), beides (C) gegen die Kontrollen --
with guarded("Transplantation"):
    tp = table("L", "transplant")
    tsum = table("L", "transplant_summary")
    modes = [m for m in ["residual", "kv", "both"] if m in set(tp["mode"])]
    controls = [c for c in ["none", "random", "identity", "permuted_alignment"] if c in set(tp["control"])]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, q, lab in zip(axes, ["fD", "logp_target"], ["f_D: deutscher Anteil der Fortsetzung", "ln p(Ziel)"]):
        piv = tp.groupby(["mode", "control"])[q].mean().unstack("control").reindex(modes)[controls]
        piv.plot(kind="bar", ax=ax, width=0.8)
        ax.set_title(lab, fontsize=9)
        ax.set_xlabel("Modus (none = Transplantat; Kontrollen: random, identity, permuted_alignment)")
        ax.legend(fontsize=7)
    savefig("transplant_modes")
    plt.show()
    if len(tsum):
        show(tsum)
        print("Vorzeichen der f_D-Verschiebung gegen die Identitätskontrolle stimmt mit der nativen Achse überein:",
              dict(zip(tsum["mode"], tsum["sign_agrees_with_native"])))


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    # failed substages (stages G-L) from the manifests: a stage whose table is an empty default
    # must not read plain "ok" in the summary (reused stages included)
    for st, r in report.items():
        if r.get("status") in ("ok", "reused") and r.get("run_dir"):
            r.update(C.substage_status(r["run_dir"], r["status"]))
    substage_errors = {st: r["substage_errors"] for st, r in report.items() if r.get("substage_errors")}
    status_overall = ("error" if any(r.get("status") == "error" for r in report.values())
                      else "substage_errors" if substage_errors else "ok")
    if substage_errors:
        print("ACHTUNG, fehlgeschlagene Substufen:", substage_errors)
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        status_overall=status_overall, substage_errors=substage_errors,
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
